# **1.Introduction:Real vs Fake image detection**




In [ ]:
import kagglehub
import os
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.datasets import ImageFolder
import torchvision.transforms as tt
from torch.utils.data import DataLoader, Subset
from torchvision.utils import make_grid
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import torch.nn.utils as nn_utils
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

dataset_=kagglehub.dataset_download('xhlulu/140k-real-and-fake-faces')
print('Dataset downloaded to:', dataset_)

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

## 2. Data loading

In [ ]:
# Locate the dataset folder that kagglehub downloaded
data_dr = os.path.join(dataset_, 'real_vs_fake', 'real-vs-fake')
print(os.listdir(data_dr))
print('Classes:', os.listdir(os.path.join(data_dr, 'train')))

In [ ]:
# Data transforms (must be defined before the datasets that use them)
IMG_SIZE = 64

train_transform = tt.Compose([
    tt.ToTensor(),
    tt.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
    tt.RandomHorizontalFlip(p=0.5),
    tt.RandomRotation(10),
    tt.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    tt.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

valid_transform = tt.Compose([
    tt.ToTensor(),
    tt.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
    tt.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
# Load the full datasets
full_train_ds = ImageFolder(os.path.join(data_dr, 'train'), train_transform)
full_valid_ds = ImageFolder(os.path.join(data_dr, 'test'), valid_transform)

# ImageFolder sorts class folders alphabetically, so by default fake=0 and real=1.
# The rest of this notebook treats 0 = Real and 1 = Fake, so remap the labels.
print('Original class_to_idx:', full_train_ds.class_to_idx)
REAL_IDX = full_train_ds.class_to_idx['real']
FAKE_IDX = full_train_ds.class_to_idx['fake']

def remap_label(y):
    return 1 if y == FAKE_IDX else 0   # 0 = Real, 1 = Fake

full_train_ds.target_transform = remap_label
full_valid_ds.target_transform = remap_label

# Balanced subsets: 750 real + 750 fake for training, same for validation
N_PER_CLASS = 750
rng = np.random.default_rng(42)

def balanced_indices(ds, n_per_class):
    real_idx = [i for i, (_, y) in enumerate(ds.samples) if y == REAL_IDX]
    fake_idx = [i for i, (_, y) in enumerate(ds.samples) if y == FAKE_IDX]
    chosen = np.concatenate([
        rng.choice(real_idx, n_per_class, replace=False),
        rng.choice(fake_idx, n_per_class, replace=False)
    ])
    return chosen.tolist()

train_ds = Subset(full_train_ds, balanced_indices(full_train_ds, N_PER_CLASS))
valid_ds = Subset(full_valid_ds, balanced_indices(full_valid_ds, N_PER_CLASS))

print(f"Training samples:   {len(train_ds)} ({N_PER_CLASS} real, {N_PER_CLASS} fake)")
print(f"Validation samples: {len(valid_ds)} ({N_PER_CLASS} real, {N_PER_CLASS} fake)")

In [ ]:
# Data loaders
batch_size = 32
train_loader = DataLoader(train_ds, batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size * 2, num_workers=2, pin_memory=True)
print(f"Train batches per epoch: {len(train_loader)}, validation batches: {len(valid_loader)}")

def show_sample_images(dataloader, num_images=8):
    images, labels = next(iter(dataloader))
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3))
    for i in range(num_images):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        img = np.clip((img * 0.5) + 0.5, 0, 1)  # Denormalize
        axes[i].imshow(img)
        axes[i].set_title('Real' if labels[i] == 0 else 'Fake')
        axes[i].axis('off')
    plt.suptitle('Sample Training Images', fontsize=14)
    plt.tight_layout()
    plt.show()

show_sample_images(train_loader)

In [ ]:

def to_device(data):
    if isinstance(data, (list, tuple)):
        return [x.to(device, non_blocking=True) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader:
    def __init__(self, dl):
        self.dl = dl

    def __iter__(self):
        for batch in self.dl:
            yield to_device(batch)

    def __len__(self):
        return len(self.dl)

train_dl = DeviceDataLoader(train_loader)
valid_dl = DeviceDataLoader(valid_loader)

## 3. Metrics and models

In [ ]:
def calculate_metrics(outputs, labels):
    preds = (outputs >= 0.5).float().cpu().detach().numpy()
    labels = labels.cpu().detach().numpy()

    cm = confusion_matrix(labels, preds)
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'confusion_matrix': cm
    }


In [ ]:
class BaseModel(nn.Module):
    def training_step(self, batch):
        images, labels = batch
        labels = labels.unsqueeze(1).float()
        out = self(images)
        loss = F.binary_cross_entropy(out, labels)
        return loss

    def validation_step(self, batch):
        images, labels = batch
        labels = labels.unsqueeze(1).float()
        out = self(images)
        loss = F.binary_cross_entropy(out, labels)
        return {
            'val_loss': loss.detach(),
            'outputs': out.detach().cpu(),
            'labels': labels.detach().cpu()
        }

    def validation_end(self, outputs):
        batch_loss = [x['val_loss'] for x in outputs]
        epoch_loss = torch.stack(batch_loss).mean()
        all_outputs = torch.cat([x['outputs'] for x in outputs])
        all_labels = torch.cat([x['labels'] for x in outputs])
        metrics = calculate_metrics(all_outputs, all_labels)

        return {
            'val_loss': epoch_loss.item(),
            'val_acc': metrics['accuracy'],
            'val_prec': metrics['precision'],
            'val_recall': metrics['recall'],
            'val_f1': metrics['f1_score'],
            'confusion_matrix': metrics['confusion_matrix']
        }


In [ ]:
class CNN(BaseModel):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # layer 1
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),

            # layer 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),

            # layer 3
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),

            # layer 4
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.fc_layers = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [ ]:
class ResNetModel(BaseModel):
    def __init__(self):
        super(ResNetModel, self).__init__()
        import torchvision.models as models
        self.backbone = models.resnet50(weights='IMAGENET1K_V1')

        # Freeze early layers
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze last 2 layers
        for param in self.backbone.layer4.parameters():
            param.requires_grad = True

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

        self.resize = tt.Resize((224, 224), antialias=True)

    def forward(self, x):
        if x.shape[-1] != 224:
            x = self.resize(x)
        return self.backbone(x)

In [ ]:


class EfficientNetModel(BaseModel):
    def __init__(self, variant='b0'):
        super(EfficientNetModel, self).__init__()
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

        self.backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

        # Freeze early layers
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze last 6 layers
        for param in self.backbone.features[-6:].parameters():
            param.requires_grad = True

        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

        self.resize = tt.Resize((224, 224), antialias=True)

    def forward(self, x):
        if x.shape[-1] != 224:
            x = self.resize(x)
        return self.backbone(x)


In [ ]:
# Model 4: DenseNet
class DenseNetModel(BaseModel):
    def __init__(self):
        super(DenseNetModel, self).__init__()
        from torchvision.models import densenet121, DenseNet121_Weights

        self.backbone = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)

        # Freeze early layers
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze last dense block
        for param in self.backbone.features.denseblock4.parameters():
            param.requires_grad = True

        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

        self.resize = tt.Resize((224, 224), antialias=True)

    def forward(self, x):
        if x.shape[-1] != 224:
            x = self.resize(x)
        return self.backbone(x)

## 4. Training utilities

In [ ]:
@torch.no_grad()
def evaluate(model, val_loader):
    model.eval()
    outputs = [model.validation_step(batch) for batch in val_loader]
    return model.validation_end(outputs)

def get_lr(optimizer):
    return optimizer.param_groups[0]['lr']

def save_checkpoint(model, optimizer, scheduler, epoch, history, checkpoint_path):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history
    }
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved at epoch {epoch+1}")

def load_checkpoint(model, optimizer, scheduler, checkpoint_path):
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        history = checkpoint['history']
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming from epoch {start_epoch}")
        return start_epoch, history
    return 0, []

In [ ]:



def train_model(model, train_loader, val_loader, epochs, max_lr,
                weight_decay=0.01, grad_clip=1.0, model_name="model"):
    """Train a single model and return history"""

    optimizer = torch.optim.AdamW(model.parameters(), max_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr, epochs=epochs, steps_per_epoch=len(train_loader)
    )

    checkpoint_path = f"{model_name}_checkpoint.pth"
    start_epoch, history = load_checkpoint(model, optimizer, scheduler, checkpoint_path)

    for epoch in range(start_epoch, epochs):
        model.train()
        train_losses = []

        with tqdm(train_loader, desc=f"{model_name} - Epoch {epoch+1}/{epochs}", unit="batch") as pbar:
            for batch in pbar:
                optimizer.zero_grad()
                loss = model.training_step(batch)
                train_losses.append(loss.item())
                loss.backward()

                if grad_clip:
                    nn_utils.clip_grad_value_(model.parameters(), grad_clip)

                optimizer.step()
                scheduler.step()
                pbar.set_postfix({'loss': loss.item(), 'lr': get_lr(optimizer)})

        # Validation
        val_result = evaluate(model, val_loader)

        # Save history
        history.append({
            'epoch': epoch,
            'train_loss': sum(train_losses)/len(train_losses),
            'val_loss': val_result['val_loss'],
            'val_acc': val_result['val_acc'],
            'val_prec': val_result['val_prec'],
            'val_recall': val_result['val_recall'],
            'val_f1': val_result['val_f1']
        })

        # Save checkpoint
        save_checkpoint(model, optimizer, scheduler, epoch, history, checkpoint_path)

        # Print progress
        print(f"{model_name} - Epoch {epoch+1}: Train Loss: {history[-1]['train_loss']:.4f}, "
              f"Val Loss: {history[-1]['val_loss']:.4f}, Val Acc: {history[-1]['val_acc']:.4f}, "
              f"Val F1: {history[-1]['val_f1']:.4f}")

    return history


## 5. Visualization

In [ ]:

# Visualization functions
def plot_training_history(histories, model_names):
    """Plot training and validation metrics for all models"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    metrics = ['train_loss', 'val_loss', 'val_acc', 'val_prec', 'val_recall', 'val_f1']
    titles = ['Training Loss', 'Validation Loss', 'Validation Accuracy',
              'Validation Precision', 'Validation Recall', 'Validation F1 Score']

    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        row, col = idx // 3, idx % 3
        for history, name in zip(histories, model_names):
            epochs = [h['epoch']+1 for h in history]
            values = [h[metric] for h in history]
            axes[row, col].plot(epochs, values, marker='o', linewidth=2, markersize=4, label=name)

        axes[row, col].set_xlabel('Epoch')
        axes[row, col].set_ylabel(metric.replace('_', ' ').title())
        axes[row, col].set_title(title)
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)

    plt.suptitle('Model Training History Comparison', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:

def plot_confusion_matrices(results, model_names):
    """Plot confusion matrices for all models"""
    fig, axes = plt.subplots(1, len(model_names), figsize=(5*len(model_names), 4))
    if len(model_names) == 1:
        axes = [axes]

    for idx, (result, name) in enumerate(zip(results, model_names)):
        cm = result['confusion_matrix']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                    xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
        axes[idx].set_title(f'{name}\nAccuracy: {result["val_acc"]:.3f}')
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')

    plt.suptitle('Confusion Matrices Comparison', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:


def plot_model_comparison(results, model_names):
    """Plot bar chart comparing model performances"""
    metrics = ['val_acc', 'val_prec', 'val_recall', 'val_f1']
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(metric_names))
    width = 0.8 / len(model_names)

    for idx, (result, name) in enumerate(zip(results, model_names)):
        values = [result[metric] for metric in metrics]
        offset = (idx - len(model_names)/2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=name)

        # Add value labels on bars
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                   f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('Metrics')
    ax.set_ylabel('Score')
    ax.set_title('Model Performance Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1.1)

    plt.tight_layout()
    plt.show()

In [ ]:
def visualize_predictions(models, model_names, val_loader, num_samples=8):
    """Visualize predictions from all models on sample images"""
    models = [m.to(device) for m in models]
    for model in models:
        model.eval()

    # Get batch of images
    images, true_labels = next(iter(val_loader))
    images = images[:num_samples].to(device)
    true_labels = true_labels[:num_samples]

    fig, axes = plt.subplots(num_samples, len(models) + 1, figsize=(3*(len(models)+1), 3*num_samples))

    for i in range(num_samples):
        # Show original image
        img = images[i].cpu().permute(1, 2, 0).numpy()
        img = np.clip((img * 0.5) + 0.5, 0, 1)
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'True: {"Real" if true_labels[i]==0 else "Fake"}')
        axes[i, 0].axis('off')

        # Get predictions from each model
        for j, (model, name) in enumerate(zip(models, model_names)):
            with torch.no_grad():
                output = model(images[i].unsqueeze(0))
                pred = (output > 0.5).float().item()
                prob = output.item()

            color = 'green' if pred == true_labels[i].item() else 'red'
            axes[i, j+1].imshow(img)
            axes[i, j+1].set_title(f'{name}\nPred: {"Real" if pred==0 else "Fake"}\nProb: {prob:.3f}',
                                   color=color)
            axes[i, j+1].axis('off')

    plt.suptitle('Model Predictions Comparison', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

## 6. Train and evaluate all models

In [ ]:
# Main training and evaluation
def main():
    # Hyperparameters
    EPOCHS = 10
    MAX_LR = 0.001
    WEIGHT_DECAY = 0.01
    GRAD_CLIP = 1.0

    # Initialize models
    print("\n" + "="*50)
    print("Initializing Models")
    print("="*50)

    models = {
        'CNN': CNN(),
        'ResNet50': ResNetModel(),
        'EfficientNet': EfficientNetModel(),
        'DenseNet': DenseNetModel()
    }

    # Move models to device
    for name in models:
        models[name] = models[name].to(device)
        print(f"{name} initialized on {device}")

    # Train models and collect histories
    histories = []
    trained_models = []
    model_names = []

    for name, model in models.items():
        print("\n" + "="*50)
        print(f"Training {name}")
        print("="*50)

        history = train_model(
            model, train_dl, valid_dl, EPOCHS, MAX_LR,
            WEIGHT_DECAY, GRAD_CLIP, model_name=name.replace(' ', '_')
        )

        histories.append(history)
        trained_models.append(model)
        model_names.append(name)

    # Evaluate all models
    print("\n" + "="*50)
    print("Final Evaluation Results")
    print("="*50)

    results = []
    for model, name in zip(trained_models, model_names):
        result = evaluate(model, valid_dl)
        results.append(result)
        print(f"\n{name}:")
        print(f"  Validation Loss: {result['val_loss']:.4f}")
        print(f"  Validation Accuracy: {result['val_acc']:.4f}")
        print(f"  Validation Precision: {result['val_prec']:.4f}")
        print(f"  Validation Recall: {result['val_recall']:.4f}")
        print(f"  Validation F1 Score: {result['val_f1']:.4f}")

    # Visualizations
    print("\n" + "="*50)
    print("Generating Visualizations")
    print("="*50)

    # Plot training histories
    plot_training_history(histories, model_names)

    # Plot confusion matrices
    plot_confusion_matrices(results, model_names)

    # Plot model comparison
    plot_model_comparison(results, model_names)

    # Visualize predictions on sample images
    visualize_predictions(trained_models, model_names, valid_dl, num_samples=6)

    # Create summary DataFrame
    summary_df = pd.DataFrame({
        'Model': model_names,
        'Accuracy': [r['val_acc'] for r in results],
        'Precision': [r['val_prec'] for r in results],
        'Recall': [r['val_recall'] for r in results],
        'F1 Score': [r['val_f1'] for r in results],
        'Validation Loss': [r['val_loss'] for r in results]
    })

    print("\n" + "="*50)
    print("Performance Summary")
    print("="*50)
    print(summary_df.to_string(index=False))

    # Find best model
    best_model_idx = np.argmax([r['val_acc'] for r in results])
    print(f"\n🏆 Best Model: {model_names[best_model_idx]}")
    print(f"   Accuracy: {results[best_model_idx]['val_acc']:.4f}")
    print(f"   F1 Score: {results[best_model_idx]['val_f1']:.4f}")

    # Save results
    summary_df.to_csv('model_comparison_results.csv', index=False)
    print("\nResults saved to 'model_comparison_results.csv'")

    return trained_models, model_names, histories, results

# Run the main function
if __name__ == "__main__":
    trained_models, model_names, histories, results = main()